In [1]:
%gui qt

Present finish error
Traceback (most recent call last):
  File "/home/kushal/venvs/fpl/lib/python3.13/site-packages/rendercanvas/core/coreutils.py", line 51, in log_exception
    yield
  File "/home/kushal/venvs/fpl/lib/python3.13/site-packages/rendercanvas/base.py", line 623, in _finish_present
    self._rc_request_paint()
    ~~~~~~~~~~~~~~~~~~~~~~^^
  File "/home/kushal/venvs/fpl/lib/python3.13/site-packages/rendercanvas/qt.py", line 414, in _rc_request_paint
    QtWidgets.QWidget.update(self)
    ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
RuntimeError: wrapped C/C++ object of type QRenderWidget has been deleted


In [1]:
import numpy as np
import fastplotlib as fpl
from fastplotlib.widgets.nd_widget import NDPositionsProcessor
import cmap
import itertools

Unable to find extension: VK_EXT_physical_device_drm


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Valid,Device,Type,Backend,Driver
✅ (default),AMD Radeon RX 570 Series (RADV POLARIS10),DiscreteGPU,Vulkan,Mesa 22.3.6
✅,NVIDIA GeForce RTX 3080,DiscreteGPU,Vulkan,575.57.08
❗ limited,"llvmpipe (LLVM 15.0.6, 256 bits)",CPU,Vulkan,Mesa 22.3.6 (LLVM 15.0.6)
❌,"AMD Radeon RX 570 Series (polaris10, LLVM 15.0.6, DRM 3.49, 6.1.0-41-amd64)",Unknown,OpenGL,4.6 (Core Profile) Mesa 22.3.6


pygfx version from git (0.9.0) and __version__ (0.16.0) don't match.
To silence this warning, use a fully namespaced name.


In [2]:
fpl.select_adapter(fpl.enumerate_adapters()[1])

In [3]:
xs = np.linspace(0, 2 * np.pi, 1_00)
ys = np.sin(xs)

l = np.column_stack([xs, ys])

data = np.repeat(l[None], 3, axis=0)
data.shape

(3, 100, 2)

In [4]:
colors = np.zeros(shape=(*data.shape[:2], 4), dtype=np.float32)
colors[0, :, (0, 1)] = np.column_stack([np.abs(ys), np.abs(np.cos(xs))]).T
colors[1, :, (1, 2)] = np.column_stack([np.abs(ys), np.abs(np.cos(xs))]).T
colors[2, :, (2, 0)] = np.column_stack([np.abs(ys), np.abs(np.cos(xs))]).T
colors[..., -1] = 1

In [5]:
markers = np.vstack([
    np.tile(list("oxs*"), int(data.shape[1] / 4)),
    np.tile(list("xs*o"), int(data.shape[1] / 4)),
    np.tile(list("s*ox"), int(data.shape[1] / 4)),
])

In [6]:
markers

array([['o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o',
        'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x',
        's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's',
        '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*',
        'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o',
        'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x',
        's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's',
        '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*'],
       ['x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x',
        's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's',
        '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*',
        'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o',
        'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x',
        's', '*', 'o', 'x', 's', '*', 'o', 'x', 's', '*', 'o', 'x', 's

In [7]:
sizes = np.zeros(shape=(data.shape[:2]), dtype=np.float32)
sizes[0] = np.abs(ys)
sizes[1] = np.abs(np.cos(xs))
sizes[2] = np.abs(np.cos(2 * xs))
sizes *= 10

In [8]:
def default_cmap_transform_each(p: int, data_slice: np.ndarray, s: slice):
    # create a cmap transform based on the `p` dim size
    n_displayed = data_slice.shape[1]

    # linspace that's just normalized 0 - 1 within `p` dim size
    return np.linspace(
        start=s.start / p,
        stop=s.stop / p,
        num=n_displayed,
        endpoint=False  # since we use a slice object for the displayed data, the last point isn't included
    )

In [9]:
from functools import partial

In [10]:
ndp = NDPositionsProcessor(
    data, 
    dims=list("lpd"), 
    spatial_dims=list("lpd"),
    index_mappings={"p": xs.searchsorted},
    display_window=np.pi / 4,
    # colors=colors,
    # markers=markers,
    # sizes=sizes,
    cmap_transform_each=partial(default_cmap_transform_each, data.shape[1])
)

In [11]:
out = ndp.get({"p": 0.1})

In [12]:
out["data"].shape

(3, 8, 2)

In [13]:
default_cmap_transform_each(100, out["data"], slice(0, 7, 1))

array([0.     , 0.00875, 0.0175 , 0.02625, 0.035  , 0.04375, 0.0525 ,
       0.06125])

In [14]:
out["cmap_transform_each"].shape

(8,)

In [15]:
out["colors"].shape

AttributeError: 'NoneType' object has no attribute 'shape'

In [ ]:
out["sizes"].shape

In [25]:
ref_ranges = {"p": (0, 2 * np.pi, 0.1)}

ndw = fpl.NDWidget(ref_ranges)

ndg = ndw[0, 0].add_nd_timeseries(
    data, 
    dims=list("lpd"), 
    spatial_dims=list("lpd"),
    index_mappings={"p": xs.searchsorted},
    # display_window=None,
    display_window=np.pi / 4,
    # cmap_each=["viridis"],
    colors=colors,
    markers=markers,
    sizes=sizes,
    graphic=fpl.ScatterStack,
    graphic_kwargs={"uniform_marker": False, "uniform_size": False, "edge_width": 0.0},
)

ndw.show()

RFBOutputContext()

/home/kushal/repos/fastplotlib/fastplotlib/widgets/nd_widget/_nd_positions/_nd_positions.py:718: UserWarning: must set `cmap_each` before `cmap_transform_each`
  warn("must set `cmap_each` before `cmap_transform_each`")
/home/kushal/repos/fastplotlib/fastplotlib/graphics/features/_base.py:19: UserWarning: casting float64 array to float32
  warn(f"casting {array.dtype} array to float32")


JupyterRenderCanvas(css_height='300.0px', css_width='500.0px')

In [39]:
out = ndg.processor.get({"p": 3.25})

In [40]:
out["data"].shape

(3, 12, 2)

In [41]:
out["markers"].shape

(3, 12)

In [42]:
out["sizes"]

array([[2.2031054 , 1.580014  , 0.95056045, 0.31727934, 0.31727934,
        0.95056045, 1.580014  , 2.2031054 , 2.8173256 , 3.4202015 ,
        4.009305  , 4.5822654 ],
       [9.754297  , 9.87439   , 9.954719  , 9.994966  , 9.994966  ,
        9.954719  , 9.87439   , 9.754297  , 9.59493   , 9.396926  ,
        9.161084  , 8.888354  ],
       [9.029265  , 9.500711  , 9.819287  , 9.979867  , 9.979867  ,
        9.819287  , 9.500711  , 9.029265  , 8.412535  , 7.6604443 ,
        6.7850943 , 5.800569  ]], dtype=float32)

In [21]:
ndg.graphic.graphics[0].sizes = 20

In [24]:
ndg.graphic.graphics[0].markers = "+"

In [15]:
ndg.graphic.graphics[0].world_object.geometry.markers.data

array([101, 101, 101, 101, 101, 101, 101], dtype=int32)

In [20]:
ndg.graphic.graphics[0].world_object.geometry.positions.data.shape

(12, 3)

In [21]:
ndg.graphic.graphics[0].world_object.geometry.markers.data.shape

(7,)

In [18]:
ndg.graphic.graphics[0].world_object.geometry.markers.data

array([226, 226, 226, 226, 226, 226, 226], dtype=int32)

In [17]:
ndg.graphic.graphics[0].world_object.geometry.markers.data[:] = 226
ndg.graphic.graphics[0].world_object.geometry.markers.update_full()

In [28]:
ndg.indices

{'p': 1.1869999647140497}

In [24]:
ndg.processor.get(ndg.indices)["data"].shape

(3, 8, 2)

In [17]:
p = 1_000

start, stop, step = 300, 500, 12

np.linspace(start / p, stop / p, num=16, endpoint=False).size

16

In [8]:
ndw.figure[0, 0].y_range = (-1.5, 2)

In [20]:
fig = fpl.Figure(shape=(2, 1))

fig[0, 0].add_image(ndg.processor.colors)
fig[1, 0].add_image(colors)

fig.show(maintain_aspect=False)

RFBOutputContext()

JupyterRenderCanvas(css_height='300.0px', css_width='500.0px')

In [21]:
(ndg.processor.colors == colors).all()

np.True_

In [29]:
ndg.processor.colors.flags.writeable = False

In [43]:
ndg.processor.sizes = sizes

In [23]:
br = np.broadcast_to(np.array(["bah"]), (1000000000000,))

In [26]:
np.array(["bah"]).flags.owndata

True

In [ ]:
br.base

In [24]:
br.flags.owndata

False

In [21]:
np.broadcast_to(np.array(["bah"]), (1000000000000,)).nbytes

12000000000000